In [16]:
import genbrain_model_3dot as model
import genbrain_smcnn_core.interpreter as smcnn
import genbrain_utils_genjax as gjutils
import numpy as np
import jax.numpy as jnp
import jax
from PIL import Image
import re
import os

In [9]:
# First we will collect the generative functions from the 3dot model. Lets start with 20 particles.
genfns = [
    model.initial_proposal,
    model.initial_model,
    model.step_proposal,
    model.step_model,
    model.obs_model,
]

num_particles = 20

In [10]:
# This function allows us to grab the raw data stored in ../data and turn it into numpy frames indicating pixel occupancy
def collect_frames(directory="../data/", file_pattern="frame-*.png"):
    frame_regex = re.compile(r"frame-(\d+)\.png")
    frames = []
    for filename in os.listdir(directory):
        match = frame_regex.match(filename)
        if match:
            frame_number = int(match.group(1))
            frames.append((frame_number, os.path.join(directory, filename)))
    frames.sort(key=lambda x: x[0])
    frame_paths = [path for _, path in frames]
    numpy_frames = []
    for path in frame_paths:
        with Image.open(path) as img:
            # this must be going downwards?
            im = img.convert("L").point(lambda p: 1 if p > 0 else 0)
            numpy_frames.append((np.transpose(np.flipud(np.array(im))) > 0).astype(int))
    return numpy_frames

In [11]:
# Here we convert digital x,y numpy frames into a spherical occupancy map (i.e. a visual angle occupancy grid)
xy_obs_frames = collect_frames()
vis_angle_observations = jax.vmap(lambda obs: model.find_occupied_2d_angles(obs))(
    jnp.array(xy_obs_frames)
)
len_sim = 5
obs_traces = model.generate_obs_traces(vis_angle_observations[0:len_sim])

In [12]:
# choose a random seed so you can repeat the experiment
key = jax.random.PRNGKey(100)
init_states_and_scores, first_step_states_and_scores, unrolled_pf = (
    gjutils.smc.run_particle_filter(
        obs_traces,
        num_particles,
        len_sim,
        genfns,
        key,
        model.translate_proposal_cm_to_model,
    )
)
# keep print of scores or not? its useful for debugging.

Values: P = [-inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf
 -inf -inf -inf -inf -inf -inf], Q = [ -8.614116 -12.240436  -8.066771  -7.606756 -10.532404 -10.07239
 -14.469076 -10.72326  -10.532404  -8.154101 -14.343352  -8.614116
  -8.154101 -10.263247 -10.532404 -10.72326   -7.606756 -10.72326
 -10.072391  -7.606756], O = [-31.190197 -34.666298 -34.666298 -34.666298 -38.142395 -38.142395
 -41.618492 -34.666298 -38.142395 -31.190197 -38.142395 -31.190197
 -31.190197 -34.666298 -38.142395 -34.666298 -34.666298 -34.666298
 -38.142395 -34.666298]
Values: P = [      -inf       -inf       -inf       -inf       -inf       -inf
       -inf -15.820416       -inf       -inf -31.632635       -inf
       -inf       -inf       -inf       -inf       -inf       -inf
       -inf       -inf], Q = [-3.0486007 -5.619448  -5.639096  -5.3669643 -4.9069514 -9.021802
 -5.6194477 -6.0991087 -3.754869  -3.0486007 -5.3928404 -5.140358
 -3.5086145 -3.5086145 -3.754869  -7.0177836 -5.86570

In [13]:
# connect to genstudio viz in ../viz folder rather than my old 3D visualizer.


In [14]:
def extract_xyz_from_smcnn(pf_results, obs, particles_to_animate):
    xyz_inferences = []
    p_scores = []
    particles_per_step = pf_results[1]
    resampler_per_step = pf_results[2]
    for step in range(len(obs)):
        particles = particles_per_step[step]
        particle_choicemaps = [
            p.choicemap for i, p in enumerate(particles) if i in particles_to_animate
        ]
        particle_scores = resampler_per_step[step].log_weights
        # note you would normally index the supports b/c
        # choicemap is an assembly index. but
        xyz = np.array(
            [model.egocentric_3d_map[cm["xyz"]] for cm in particle_choicemaps]
        )
        xyz_inferences.append(xyz)
        p_scores.append(particle_scores)
    return np.array(xyz_inferences), p_scores


model_variables = [
    {
        "variable": "lights",
        "parents": [],
        "support": model.bool_support,
        "subtraced": [],
    },
    {"variable": "diam", "parents": [], "support": model.diams, "subtraced": []},
    {"variable": "v3d", "parents": [], "support": model.xyz_vels, "subtraced": []},
    {
        "variable": "xyz",
        "parents": ["v3d"],
        "support": model.xyz_point_cloud,
        "subtraced": [],
    },
    {
        "variable": "ego_pos",
        "parents": ["xyz"],
        "support": model.egocentric_3d_map,
        "subtraced": [],
    },
]

proposal_variables = [
    {
        "variable": "ego_pos",
        "parents": [],
        "support": model.egocentric_3d_map,
        "subtraced": [],
    },
    {
        "variable": "xyz",
        "parents": ["ego_pos"],
        "support": model.xyz_point_cloud,
        "subtraced": [],
    },
    {"variable": "v3d", "parents": ["xyz"], "support": model.xyz_vels, "subtraced": []},
    {
        "variable": "diam",
        "parents": ["ego_pos"],
        "support": model.diams,
        "subtraced": [],
    },
    {
        "variable": "lights",
        "parents": [],
        "support": model.bool_support,
        "subtraced": [],
    },
]

obs_variables = [
    {
        "variable": "obs",
        "parents": [],
        "support": model.egocentric_2d_map,
        "subtraced": [],
    }
]

variables = [model_variables, proposal_variables, obs_variables]
assembly_size = 10

num_particles = 2

pf_results = smcnn.run_smcnn_particle_filter(
    variables,
    model.initial_model,
    model.step_model,
    model.initial_proposal,
    model.step_proposal,
    model.obs_model,
    assembly_size,
    num_particles,
    vis_angle_observations,
    "analog",
)

TypeError: run_smcnn_particle_filter() takes 9 positional arguments but 10 were given

Values: P = [      -inf       -inf       -inf -14.069142       -inf       -inf
 -31.47956  -32.92604        -inf -18.1999   -15.98837  -13.67204
 -14.069142 -20.997154 -12.622659 -24.054615       -inf       -inf
       -inf       -inf], Q = [ -3.754869   -3.754869   -9.319347   -5.619448   -7.2321143 -10.469468
  -9.362024   -7.25119    -3.6446338  -5.6194477  -5.159434   -5.6194477
  -5.619448   -7.711205   -5.6194477  -7.0030193  -4.8921866  -3.508615
  -3.508615   -5.140358 ], O = [-34.666298 -34.666298 -38.142395 -38.142395 -38.142395 -31.190197
 -48.57069  -41.618492 -38.142395 -38.142395 -38.142395 -38.142395
 -38.142395 -41.618492 -38.142395 -38.142395 -31.190197 -31.190197
 -31.190197 -34.666298]
Values: P = [-15.793759 -18.565975       -inf       -inf       -inf       -inf
       -inf       -inf       -inf -12.996509       -inf -22.79751
       -inf       -inf -12.996509       -inf -12.025179 -21.972832
       -inf       -inf], Q = [ -7.00302    -5.366965   -5.0282054  -5.6194